# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata about the dataset
print(f"Dataset Name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"License: {dataset.metadata.license}\n")
print(f"Published: {dataset.metadata.datePublished}\n")
print(f"Coverage: {dataset.metadata.spatialCoverage}, {dataset.metadata.temporalCoverage}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Inspect the dataset structure using the `mlcroissant` library API:

In [ ]:
# List record sets and fields by their @id
record_sets = list(dataset.record_sets())
print("Record Sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', rs['@id'])}")

# For demonstration, show fields for the first record set
if record_sets:
    record_set_id = record_sets[0]['@id']
    fields = dataset.fields(record_set=record_set_id)
    print(f"\nFields for Record Set '{record_set_id}':")
    for field in fields:
        field_id = field.get('@id', 'unknown')
        field_name = field.get('name', field_id)
        field_type = field.get('dataType', '')
        print(f"- {field_id}: {field_name}, Type: {field_type}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s from the overview.

In [ ]:
# Extract records for each record set

dataframes = {}

for rs in record_sets:
    rs_id = rs['@id']
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nLoaded DataFrame for record set {rs_id} (rows: {len(df)}, columns: {df.columns.tolist()})")
    else:
        print(f"\nNo records loaded for record set {rs_id}")

# For demonstration, show the head of the first available DataFrame
demo_rs_id = next(iter(dataframes.keys()), None)
if demo_rs_id:
    print(f"\nPreview of data for record set {demo_rs_id}:")
    display(dataframes[demo_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data by key attributes.

We'll demonstrate filtering and normalization for a numeric field in one record set. _Please replace the placeholder field IDs with values from your record set structure if necessary._

In [ ]:
# Example EDA: Select numeric field for analysis using @id
# Identify available numeric fields

if demo_rs_id:
    df = dataframes[demo_rs_id]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields detected: {numeric_fields}")

    # Choose the first numeric field for demo
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[numeric_field_id + '_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Group by another field, if available
        group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id}:")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields.

In [ ]:
# Basic visualization (requires matplotlib)
import matplotlib.pyplot as plt

if demo_rs_id and numeric_fields:
    df = dataframes[demo_rs_id]
    plt.figure(figsize=(8, 5))
    plt.hist(df[numeric_field_id].dropna(), bins=20, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id} in {demo_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field exists, show boxplot
    if group_fields:
        group_field_id = group_fields[0]
        plt.figure(figsize=(10, 6))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides ordered logistic regression outputs and survey-based records on household adoption of indigenous and modern knowledge for rangeland management.
- We loaded the dataset, examined available record sets and fields (referenced by their `@id`), and demonstrated filtering, normalization, grouping, and visualization of numeric fields.
- Further analyses can be undertaken by referencing specific record sets or fields using their `@id` and leveraging the rich metadata via Croissant.